# Signed Jumps 04b — real-time seconds trigger

Protocol: `discussions/Signed_Jumps/009_claude_sj04b_preregistration.md` (fixed before this run).

Trigger at the first second where the trailing-60 s return exceeds c × the SJ 01 engine's 1-minute σ (past data only), then measure the signed return from trigger + δ to trigger + δ + H. Gate: c = 4, δ = 2 s, H = 60 s, gross q2.5 > 0.

In [ ]:
import os,sys,json,time,importlib,subprocess
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
from dotenv import load_dotenv
from dataclasses import replace
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'));load_dotenv(ROOT/'.env')
from data_engineering.modeling import signed_jumps as sj, signed_jumps_seconds as ss, signed_jumps_trigger as st
for m in (sj,ss,st):importlib.reload(m)
assert st.VERSION=='SJ04b-0.1.0'
SYNTHETIC=os.getenv('SJ04B_SYNTHETIC','0')=='1'
p=sj.Params();tp=st.TrigParams()
if SYNTHETIC:
    from tempfile import TemporaryDirectory
    tmp=TemporaryDirectory(prefix='sj04b-');OUT=Path(tmp.name)/'05'
    p=sj.Params(n_boot=200,half_lives=(1,));tp=st.TrigParams(n_boot=300)
else:
    OUT=ROOT/'notebooks'/'Signed_Jumps'/'05'
OUT.mkdir(parents=True,exist_ok=True)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
if not SYNTHETIC:
    chk=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_signed_jumps_trigger.py')],cwd=str(ROOT),env=env,capture_output=True,text=True)
    print(chk.stdout[-600:]);print(chk.stderr[-400:]);assert chk.returncode==0,'tests failed'
print(json.dumps(st.registration(tp),indent=1))

## Engine σ (SJ 01, unchanged) and 1-second data

In [ ]:
T0=time.time()
if SYNTHETIC:
    secs,one=ss.simulate_seconds(n_days=14,n_assets=len(p.assets),seed=4,drift_bp=6,drift_s=60)
    secs={a:v for a,v in zip(p.assets,secs.values())};one={a:v for a,v in zip(p.assets,one.values())}
    assets,pan,_=sj.prepare(one,p)
else:
    BIN=Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent/'Binance'
    S={A:sj.load_spot_1m(BIN/'basis'/'spot'/f'{A}USDT_spot_1m.parquet') for A in p.assets}
    assets,pan,_=sj.prepare(S,p)
    secs={A:ss.load_spot_1s(sorted((BIN/'spot_1s'/f'{A}USDT').glob(f'{A}USDT_spot_1s_*.parquet'))) for A in p.assets}
sig={a.name:st.sigma_minute(a) for a in assets}
for a in assets:print(a.name,'median 1-min sigma (bp):',round(float(sig[a.name].loc[secs[a.name].index.min():secs[a.name].index.max()].median()*1e4),2))
print(f'{time.time()-T0:.0f}s')

## Triggers, forward returns, grid

In [ ]:
tabs=[];evs={}
for c in tp.c_grid:
    parts=[]
    for a in assets:
        tr=st.triggers(secs[a.name],sig[a.name],c,tp)
        f=st.forward(tr,secs[a.name],tp);f['asset']=a.name;parts.append(f)
    ev=pd.concat(parts,ignore_index=True);evs[c]=ev
    tabs.append(st.grid_table(ev,tp,c))
tab=pd.concat(tabs,ignore_index=True);tab.to_csv(OUT/'05_grid.csv',index=False)
evs[tp.c_trig].to_csv(OUT/'05_triggers_c4.csv.gz',index=False)
print('triggers per c:',{c:len(e) for c,e in evs.items()})
for c in tp.c_grid:
    print(f'c = {c}: gross bp (rows = latency delta s, cols = hold H s)')
    display(tab[tab.c_trig==c].pivot_table(index='delta_s',columns='h_s',values='gross_bp').round(2))
print('c = 4: 95% lower bounds');display(tab[tab.c_trig==tp.c_trig].pivot_table(index='delta_s',columns='h_s',values='lo').round(2))
print('c = 4: share positive');display(tab[tab.c_trig==tp.c_trig].pivot_table(index='delta_s',columns='h_s',values='share_positive').round(3))

## Splits (c = 4, δ = 2 s, H = 60 s)

In [ ]:
e=evs[tp.c_trig];k=f'g_{tp.delta_s}_{tp.h_s}'
day=pd.DatetimeIndex(e['tau']).normalize().asi8
rows=[]
for name,m in [('all',np.ones(len(e),bool))]+[(f'asset={a}',(e.asset==a).to_numpy()) for a in p.assets]+[('dir=up',(e.dir>0).to_numpy()),('dir=down',(e.dir<0).to_numpy())]:
    mm,lo,hi,n=st.boot_mean(e[k].to_numpy()[m],day[m],tp.n_boot,5)
    rows.append({'split':name,'gross_bp':mm,'lo':lo,'hi':hi,'n':n})
spl=pd.DataFrame(rows);display(spl.round(2));spl.to_csv(OUT/'05_splits.csv',index=False)
print('trigger z / R60 (bp) quantiles:');display(e[['z','R60_bp']].abs().quantile([.1,.5,.9]).round(2))

## Decision

In [ ]:
dec=st.decide(tab,tp);dec.update(triggers={str(c):int(len(v)) for c,v in evs.items()},synthetic=SYNTHETIC)
print(json.dumps(dec,indent=1,default=float))
if not SYNTHETIC:
    (OUT/'05_decision.json').write_text(json.dumps(dec,indent=1,default=float))
    (OUT/'05_manifest.json').write_text(json.dumps({'registration':st.registration(tp),'runtime_s':time.time()-T0},indent=1,default=str))
print(f'total {time.time()-T0:.0f}s')